# IMPORTS

In [ ]:
# For calculation and data visulization
import numpy as np
import pandas as pd
import random
from collections import Counter

# we train and then we split and then we train and then we split and then we train and split, and believe it or not we train and split and at the end we test
from sklearn.model_selection import train_test_split

# importing the feature extraction
from sklearn.feature_extraction.text import TfidfVectorizer

# importing baseline
from sklearn.dummy import DummyClassifier

# importing the scikit models and calculations
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix

# importing the tokenization
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt_tab')

# for DistilBERT model
import torch

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)


In [ ]:
print("hello Sheldon")
results = []

In [ ]:
# Easier to Visualize the data and to see that all the tags are okay
# Easier implementation of the tsv file

from google.colab import files
uploaded = files.upload()


df = pd.read_csv("Group4_anno.tsv", sep="\t")
df.head()

# Maybe pandas was taking nan as empty so it is not helping at all
df["LABEL"] = df["LABEL"].fillna("NaN")

df["LABEL"] = df["LABEL"].str.lower()
print(df["LABEL"].value_counts())

TOKENIZER AND VECTORISER


In [ ]:
""" gettign the datas from the csv filee"""
def get_file(file_nem):
  with open(file_nem) as f:
    #print(file_nem)
    loines = f.readlines()
    return loines[1:] #without titles

def get_datas(lines):
  test_data = []
  train_data = []
  dev_data = []
  for i in range(len(lines)):
    #print(lines[i])
    line = lines[i].strip("\t\n") #remove all extra tabs and the \n
    bits = line.split("\t") #:)
    if len(bits) == 1: #if the line is empty we skip
      continue
    #print(bits)
    #input()

    # we get the labels equal all of them
    label = bits[3].lower()

    text = bits[4]
    typee = bits[1]
    data = [text,label] #current format, but can be changed (good for puttign through the scrambler tho)
    if typee == "Exploration":
      #print(data)
      dev_data.append(data)
      continue #w/o this it cuts off the text??? how??? whyy???
    #if "scene" in data[0]: #scene lines urghhh
      #data[0] = data[0][11:]
    #else:
      #data[0] = data[0][14:].strip('" ') #sooo, the joint and additional lines have the person bit baked in, so gotta remove it

    if typee == "Joint-Consolidated": #the joint data aka test data
      test_data.append(data)
      #print(data)
    elif typee == "additional": #our train data
      #data[0] = data[0][14:].strip('" ')
      #print(data)
      train_data.append(data)
  return train_data,dev_data,test_data
def get_scrambled(lst_labeled_data): #THE SCRAMBLEERRRRRRRRR (from ML1)
  """
  This function takes a list of all files and scrambles its order.
  """
  smashed_lst = list()
  rangee = range(len(lst_labeled_data))
  for i in rangee:
      one = random.choice(lst_labeled_data)
      lst_labeled_data.remove(one) #we remove so we dont choose the same twice
      smashed_lst.append(one)
  return smashed_lst

def get_xy(data_pairs): #im too lazy to find a better way
  x = []
  y = []
  for i in range(len(data_pairs)):
    pair = data_pairs[i]
    x.append(pair[0])
    y.append(pair[1])
  return x,y

file_nem = "Group4_anno.tsv"
lines = get_file(file_nem)
train_data,dev_data,test_data = get_datas(lines)
#this is awful im sorryyyy
scrambled_train_pairs = get_scrambled(train_data)
scrambled_dev_pairs = get_scrambled(dev_data)
scrambled_test_pairs = get_scrambled(test_data)

X_train,y_train = get_xy(scrambled_train_pairs)
X_dev,y_dev = get_xy(scrambled_dev_pairs)
X_test,y_test = get_xy(scrambled_test_pairs)

X_raw = X_train + X_dev + X_test
y = y_train + y_dev + y_test
assert len(X_raw) == len(y) #checkinggg
print(X_raw[510])
print(y[510])



In [ ]:

class NonCountVectorizer: #from ML1:)
  """
  This is the vectorizer that turns X_raw into bag-o-words vectors!
  """
  def fit(self,X_data): #this is like a faux constructor, which makes the vector structure.
    self.X_data = X_data
    self.ordered_vocabulary = self.vocabulator(X_data)
    self.ordered_dict = self.dictionulator()


  def tokenizer(self,stwing):
    """takes a string, a review, as input and outputs its tokens"""
    #print(stwing)
    #input()
    return word_tokenize(stwing.lower()) #eyyy its nltk now! also lowering to avoid different words that are the same


  def vocabulator(self,X_data):
    """takes a list of raw reviews and, for each review, tokenizes it and add the tokens to a set, which becomes the vocabulary for all reviews."""
    vocab = set()
    for i in range(len(X_data)):
        words = self.tokenizer(X_data[i])
        for word in words:
            vocab.add(word) #using a set removes duplicate words.
    ordered_vocabulary = list(vocab) #as we need the index of each item later, we turn the vocabulary into a list.
    return ordered_vocabulary

  def dictionulator(self):
    """
    takes a list of unique words, and creates a dictionary with the keys as the word and value as each words' index. Makes the vectorizer faster
    """
    ordered_dict = dict()
    for i, word in enumerate(self.ordered_vocabulary):
      ordered_dict[word] = i
    return ordered_dict


  def transform(self,X_data):
    """
    transform inputed data X_train into bag-o-words vectors, based on the fit method.
    """

    X = np.zeros((len(X_data),len(self.ordered_dict)))

    for i,doc in enumerate(X_data):
        tokens = self.tokenizer(doc)
        for word in tokens:
            try: #we could also check if the input has already happened, but I think that would be worse time wise, and readability wise
              X[i,self.ordered_dict[word]] = 1
            except: #if word isnt in dict then we skip
              continue
    return X
vectorizer = NonCountVectorizer()
vectorizer.fit(X_raw) #creates dictionary and ordered vocabulary

""" OUR BOW VECTORS ARE HERE"""
X_train_vector = vectorizer.transform(X_train)
X_dev_vector = vectorizer.transform(X_dev)
X_test_vector = vectorizer.transform(X_test)

print(X_train_vector.shape)
print(X_dev_vector.shape)
print(X_test_vector.shape)

# scikit-learn TFIDF

In [ ]:
# We create it like this to avoid manual work thus faster
tfidf = TfidfVectorizer(tokenizer = word_tokenize)


# Here we pass the different parts of the dataset to not have leakage
X_traintf = tfidf.fit_transform(X_train)
X_devtf = tfidf.transform(X_dev)
X_testtf = tfidf.transform(X_test)

print("Train TF-IDF shape:", X_traintf.shape)
print("Test TF-IDF shape:", X_testtf.shape)

# Majority Tag Baseline:

In [ ]:
# Here we call our baseline with an import so faster / better
majority_tag = DummyClassifier(strategy="most_frequent")

# We fit the train data to work with
majority_tag.fit(X_traintf, y_train)

# and get the score to see how it behaves in each model
train_score = majority_tag.score(X_traintf, y_train)

# We append the data into our results table
print("Train score:", train_score)
results.append({"Model": "Baseline",
               "Feature_extraction": "Nan",
                "Dataset": "Train",
               "Accuracy": train_score,
                })

# We do the same for dev and test
dev_score = majority_tag.score(X_devtf, y_dev)

print("Dev score:", dev_score)
results.append({"Model": "Baseline",
               "Feature_extraction": "Nan",
                "Dataset": "Dev",
               "Accuracy": dev_score,
                })

test_score = majority_tag.score(X_testtf, y_test)

print("Test score:", test_score)
results.append({"Model": "Baseline",
               "Feature_extraction": "Nan",
                "Dataset": "Test",
               "Accuracy": test_score,
                })

to change from bow to tfidf:

bow:

*   X_train_vector,
*   X_dev_vector
*   X_test_vector


tfidf:

*   X_traintf,
*   X_devtf,
*   X_testtf










X_testtf

# Model1: SVM

In [ ]:
svm_model = LinearSVC(random_state=0).fit(X_traintf, y_train)
score_train = svm_model.score(X_traintf, y_train)



print("Train score:", score_train)
results.append({
    "Model": "SVM",
    "Feature_extraction": "TFIDF",
    "Dataset": "Train",
    "Accuracy": score_train,
})

#run the same for the developer data
svm_dev = svm_model.predict(X_devtf)
score_dev = svm_model.score(X_devtf, y_dev)

print("Dev score:", score_dev)
results.append({
    "Model": "SVM",
    "Feature_extraction": "TFIDF",
    "Dataset": "Dev",
    "Accuracy": score_dev,
})

#run it on the test data
svm_test = svm_model.predict(X_testtf)
score_test = svm_model.score(X_testtf, y_test)

y_pred_test = svm_model.predict(X_testtf)

print("Test score:", score_test)
results.append({
    "Model": "SVM",
    "Feature_extraction": "TFIDF",
    "Dataset": "Test",
    "Accuracy": score_test,
     "F1": f1_score(y_test, y_pred_test, average="macro")
})

## Model 2: Logistic Regression

In [ ]:
# Here we get the logistic regression for the train data of tfidf X_traintf, X_devtf , X_testtf
# for bow : X_train_vector , X_dev_vector, X_test_vector
clf = LogisticRegression(random_state=0).fit(X_traintf, y_train)
score_train = clf.score(X_traintf, y_train)

y_train_pred = clf.predict(X_traintf)

print("Train score:", score_train)
results.append({
               "Model": "Logistic Regression",
               "Feature_extraction": "TFIDF",
               "Dataset": "Train",
               "Accuracy": score_train,
               "F1": f1_score(y_train, y_train_pred, average="macro")
                })

# run the same for the developer data
clf_dev = clf.predict(X_devtf)
score_dev = clf.score(X_devtf, y_dev)


print("Dev score:", score_dev)
results.append({
               "Model": "Logistic Regression",
               "Feature_extraction": "TFIDF",
               "Dataset": "Dev",
               "Accuracy": score_dev,

                })

# And finally we run it on the test data
clf_test =clf.predict(X_testtf)
score_test = clf.score(X_testtf, y_test)


# in all of the runs the result will be saved in the results list so we can make a table later
print("Test score:", score_test)
results.append({
               "Model": "Logistic Regression",
               "Feature_extraction": "TFIDF",
               "Dataset": "Test",
               "Accuracy": score_test,
                })

## Model 3: DistilBERT

In [ ]:
# prepare training and testing data

# label mapping
label2id = {
    "tact": 0,
    "agreement": 1,
    "approbation": 2,
    "sympathy": 3,
    "n/a": 4
}
id2label = {v: k for k, v in label2id.items()}

# convert string labels to integer ids
y_train_ids = [label2id[y] for y in y_train]
y_dev_ids = [label2id[y] for y in y_dev]
y_test_ids = [label2id[y] for y in y_test]

# build HuggingFace datasets
train_dataset = Dataset.from_dict({
    "text": X_train,
    "label": y_train_ids
})

dev_dataset = Dataset.from_dict({
    "text": X_dev,
    "label": y_dev_ids
})

test_dataset = Dataset.from_dict({
    "text": X_test,
    "label": y_test_ids
})

dataset = DatasetDict({
    "train": train_dataset,
    "validation": dev_dataset,
    "test": test_dataset
})
print(dataset)

In [ ]:
# tokenization

# load DistilBERT tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# tokenize the text
def tokenize_function(examples):
  '''tokenize the text'''
  return tokenizer(
      examples["text"],
      truncation=True,
      padding="max_length",
      max_length=128
  )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# remove raw text column and convert data to PyTorch format
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset.set_format("torch")

print(tokenized_dataset)

In [ ]:
import numpy as np
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# define model_name here as it was causing a NameError
model_name = "distilbert-base-uncased"

# label mapping (re-defined for self-containment)
label2id = {
    "tact": 0,
    "agreement": 1,
    "approbation": 2,
    "sympathy": 3,
    "n/a": 4
}
id2label = {v: k for k, v in label2id.items()}

# run the DistilBERT classification model

# load the model
distilbert = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5,
    id2label=id2label
)
# define evaluation methods
def compute_metrics(eval_pred):
  '''compute accuracy and macro-F1'''
  logits, labels = eval_pred
  preds = np.argmax(logits, axis=-1)
  acc = accuracy_score(labels, preds)
  macro_f1 = f1_score(labels, preds, average="macro")
  return {
      "accuracy": acc,
      "macro_f1": macro_f1  # macro-F1 score is more suitable for this task
  }

# set training arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",  # evaluate every epoch
    save_strategy="epoch", # set save strategy to epoch to match eval strategy
    num_train_epochs=10,
    per_device_train_batch_size=8,
    load_best_model_at_end=True, # This line ensures the best performing model on the validation set is loaded at the end of training
    metric_for_best_model='macro_f1', # This line specifies that macro F1 is used to determine the best model
    greater_is_better=True
    # Removed report_to="none" to enable logging
)

# creat trainer
trainer = Trainer(
    model=distilbert,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics
)

# start training
trainer.train()


In [ ]:
# evaluate on dev and test sets
dev_results = trainer.evaluate(tokenized_dataset["validation"])
test_results = trainer.evaluate(tokenized_dataset["test"])
print("Validation results:")
print(dev_results)
print("\nTest results:")
print(test_results)

In [ ]:
# results of DistilBERT

# train set
train_pred_output = trainer.predict(tokenized_dataset["train"])
y_train_pred = np.argmax(train_pred_output.predictions, axis=-1)

train_acc = accuracy_score(y_train_ids, y_train_pred)
train_macro_f1 = f1_score(y_train_ids, y_train_pred, average="macro")

print("Train Accuracy:", train_acc)
print("Train Macro-F1:", train_macro_f1)

results.append({
    "Model": "DistilBERT",
    "Feature_extraction": "Contextual",
    "Dataset": "Train",
    "Accuracy": train_acc,
    "Macro_F1": train_macro_f1
})

# dev set
dev_pred_output = trainer.predict(tokenized_dataset["validation"])
y_dev_pred = np.argmax(dev_pred_output.predictions, axis=-1)

dev_acc = accuracy_score(y_dev_ids, y_dev_pred)
dev_macro_f1 = f1_score(y_dev_ids, y_dev_pred, average="macro")

print("Dev Accuracy:", dev_acc)
print("Dev Macro-F1:", dev_macro_f1)

results.append({
    "Model": "DistilBERT",
    "Feature_extraction": "Contextual",
    "Dataset": "Dev",
    "Accuracy": dev_acc,
    "Macro_F1": dev_macro_f1
})

# test set
test_pred_output = trainer.predict(tokenized_dataset["test"])
y_test_pred = np.argmax(test_pred_output.predictions, axis=-1)

test_acc = accuracy_score(y_test_ids, y_test_pred)
test_macro_f1 = f1_score(y_test_ids, y_test_pred, average="macro")

print("Test Accuracy:", test_acc)
print("Test Macro-F1:", test_macro_f1)

results.append({
    "Model": "DistilBERT",
    "Feature_extraction": "Contextual",
    "Dataset": "Test",
    "Accuracy": test_acc,
    "Macro_F1": test_macro_f1
})


# Result Table:

In [ ]:
final_results = []

def add_result(model, feature_extraction, dataset, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")

    final_results.append({
        "Model": model,
        "Feature_extraction": feature_extraction,
        "Dataset": dataset,
        "Accuracy": acc,
        "Macro_F1": macro_f1
    })

# majority baseline
majority_label = Counter(y_train_ids).most_common(1)[0][0]

y_pred_train_majority = np.full(len(y_train_ids), majority_label)
y_pred_dev_majority   = np.full(len(y_dev_ids), majority_label)
y_pred_test_majority  = np.full(len(y_test_ids), majority_label)

add_result("Majority", "-", "Train", y_train_ids, y_pred_train_majority)
add_result("Majority", "-", "Dev",   y_dev_ids,   y_pred_dev_majority)
add_result("Majority", "-", "Test",  y_test_ids,  y_pred_test_majority)

# SVM model (BOW)
svm_model_bow = LinearSVC(random_state=0).fit(X_train_vector, y_train)
y_train_pred_bow_str = svm_model_bow.predict(X_train_vector)
y_dev_pred_bow_str   = svm_model_bow.predict(X_dev_vector)
y_test_pred_bow_str  = svm_model_bow.predict(X_test_vector)

y_train_pred_bow = [label2id[label] for label in y_train_pred_bow_str]
y_dev_pred_bow   = [label2id[label] for label in y_dev_pred_bow_str]
y_test_pred_bow  = [label2id[label] for label in y_test_pred_bow_str]

add_result("SVM", "BOW", "Train", y_train_ids, y_train_pred_bow)
add_result("SVM", "BOW", "Dev",   y_dev_ids,   y_dev_pred_bow)
add_result("SVM", "BOW", "Test",  y_test_ids,  y_test_pred_bow)

# SVM model (TF-IDF)
svm_model_tfidf = LinearSVC(random_state=0).fit(X_traintf, y_train)
y_train_pred_tfidf_str = svm_model_tfidf.predict(X_traintf)
y_dev_pred_tfidf_str   = svm_model_tfidf.predict(X_devtf)
y_test_pred_tfidf_str  = svm_model_tfidf.predict(X_testtf)

y_train_pred_tfidf = [label2id[label] for label in y_train_pred_tfidf_str]
y_dev_pred_tfidf   = [label2id[label] for label in y_dev_pred_tfidf_str]
y_test_pred_tfidf  = [label2id[label] for label in y_test_pred_tfidf_str]

add_result("SVM", "TF-IDF", "Train", y_train_ids, y_train_pred_tfidf)
add_result("SVM", "TF-IDF", "Dev",   y_dev_ids,   y_dev_pred_tfidf)
add_result("SVM", "TF-IDF", "Test",  y_test_ids,  y_test_pred_tfidf)

# logistic regression model (BOW)
clf_bow = LogisticRegression(random_state=0).fit(X_train_vector, y_train)
y_train_pred_bow_lr_str = clf_bow.predict(X_train_vector)
y_dev_pred_bow_lr_str   = clf_bow.predict(X_dev_vector)
y_test_pred_bow_lr_str  = clf_bow.predict(X_test_vector)

y_train_pred_bow_lr = [label2id[label] for label in y_train_pred_bow_lr_str]
y_dev_pred_bow_lr   = [label2id[label] for label in y_dev_pred_bow_lr_str]
y_test_pred_bow_lr  = [label2id[label] for label in y_test_pred_bow_lr_str]

add_result("Logistic Regression", "BOW", "Train", y_train_ids, y_train_pred_bow_lr)
add_result("Logistic Regression", "BOW", "Dev",   y_dev_ids,   y_dev_pred_bow_lr)
add_result("Logistic Regression", "BOW", "Test",  y_test_ids,  y_test_pred_bow_lr)

# logistic regression model (TF-IDF)
clf_tfidf = LogisticRegression(random_state=0).fit(X_traintf, y_train)
y_train_pred_tfidf_lr_str = clf_tfidf.predict(X_traintf)
y_dev_pred_tfidf_lr_str   = clf_tfidf.predict(X_devtf)
y_test_pred_tfidf_lr_str  = clf_tfidf.predict(X_testtf)

y_train_pred_tfidf_lr = [label2id[label] for label in y_train_pred_tfidf_lr_str]
y_dev_pred_tfidf_lr   = [label2id[label] for label in y_dev_pred_tfidf_lr_str]
y_test_pred_tfidf_lr  = [label2id[label] for label in y_test_pred_tfidf_lr_str]

add_result("Logistic Regression", "TF-IDF", "Train", y_train_ids, y_train_pred_tfidf_lr)
add_result("Logistic Regression", "TF-IDF", "Dev",   y_dev_ids,   y_dev_pred_tfidf_lr)
add_result("Logistic Regression", "TF-IDF", "Test",  y_test_ids,  y_test_pred_tfidf_lr)

# DistilBERT
train_pred_output = trainer.predict(tokenized_dataset["train"])
y_train_pred = np.argmax(train_pred_output.predictions, axis=-1)

dev_pred_output = trainer.predict(tokenized_dataset["validation"])
y_dev_pred = np.argmax(dev_pred_output.predictions, axis=-1)

test_pred_output = trainer.predict(tokenized_dataset["test"])
y_test_pred = np.argmax(test_pred_output.predictions, axis=-1)

add_result("DistilBERT", "Contextual", "Train", y_train_ids, y_train_pred)
add_result("DistilBERT", "Contextual", "Dev",   y_dev_ids,   y_dev_pred)
add_result("DistilBERT", "Contextual", "Test",  y_test_ids,  y_test_pred)

# data frame
results_df = pd.DataFrame(final_results)

# sort order
dataset_order = ["Train", "Dev", "Test"]
model_order = ["Majority", "Logistic Regression", "SVM", "DistilBERT"]

results_df["Dataset"] = pd.Categorical(results_df["Dataset"], categories=dataset_order, ordered=True)
results_df["Model"] = pd.Categorical(results_df["Model"], categories=model_order, ordered=True)

results_df = results_df.sort_values(["Dataset", "Model"])

print(results_df)

In [ ]:
results_table = pd.DataFrame(results)

final_table = results_table.pivot_table(
    values=["Accuracy", "F1"],
    index="Dataset",
    columns=["Model", "Feature_extraction"]
)

print("\nFinal Results Table:")
print(final_table)